In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

import sys
sys.path.append('../../../1_figure_CL_proof_of_concept/code/')
import utils_00 as gf_utils
import os

large_data_dir = gf_utils.large_data_dir


In [ ]:
def get_gapfill_likelihoods(gapfill,error_rate_dicts,alleles):
    likelihoods = gf_utils.get_likelihoods_of_true_allele(gapfill, error_rate_dicts)
    likelihood_list = {}
    for allele in alleles:
        if allele in likelihoods:
            likelihood_list[allele] = likelihoods[allele]
        else:
            likelihood_list[allele] = 10e-10
    return likelihood_list
error_rate_dir = 'data/error_rate_dicts'
error_rate_dicts = gf_utils.get_error_rate_dicts(error_rate_dir)

In [3]:
feature_set = 'feature_set_all.csv'


# For every observed gapfill, compute likelihood under possible alleles in feature set

In [ ]:
lib = 'GBM'
BC = 'BC002'

### first get probe_reads to use for the patient
adata_path = large_data_dir + 'GBM_' + BC + '_genotyped.h5ad' ### this is just for cell names; it would not usually use the genotyped h5ad because it wouldn't be generated yet, but cell names are the same so we use it here to not have to upload a redundant h5ad just for this step
gf_dir = large_data_dir + 'gf_decrosslink_4plex/BC' + str(int(BC.replace('BC',''))) + '_giftwrap/'
            
min_percent_supporting = 0.9
collapse_across_probes = True
probe_reads = gf_utils.get_input_probe_reads(gf_dir, read_threshold = 0, adata_path = adata_path, min_percent_supporting=min_percent_supporting, collapse_across_probes=collapse_across_probes)

sample = BC + '_' + lib

alt_gapfill_set = pd.read_csv(feature_set)

sample = BC + '_' + lib
alt_gapfill_set = alt_gapfill_set.loc[alt_gapfill_set['sample'] == sample]

if len(alt_gapfill_set) > 0:
    ## reformat alt gapfill set
    manifest = gf_utils.get_manifest(gf_dir)
    probe_name_to_probe_idx = dict(zip(manifest['name'], manifest['index']))
    alt_gapfill_set['probe_idx'] = alt_gapfill_set['original_name'].map(probe_name_to_probe_idx)
    
    alt_gapfill_set = pd.concat([
        alt_gapfill_set[['probe_idx', 'name', 'gapfill']].rename(columns={'gapfill': 'gapfill_value'}).assign(variant=True),
        alt_gapfill_set[['probe_idx', 'name', 'gapfill_from_transcriptome']].rename(columns={'gapfill_from_transcriptome': 'gapfill_value'}).assign(variant=False)
    ], ignore_index=True)

    alt_gapfill_set = alt_gapfill_set.dropna(subset='probe_idx')
    # print(alt_gapfill_set)
    
    ### from now on, we only need to consider probes that have an alt gapfill
    probe_reads = probe_reads.loc[probe_reads['probe_idx'].isin(alt_gapfill_set['probe_idx'])]

    unique_gapfill_list = probe_reads[['probe_idx','gapfill']].drop_duplicates()

    possible_alleles = {}
    for probe_idx in probe_reads['probe_idx'].unique():
        possible_alleles[probe_idx] = alt_gapfill_set.loc[alt_gapfill_set['probe_idx'] == probe_idx, 'gapfill_value'].unique()

    unique_gapfill_list['likelihood'] = unique_gapfill_list.apply(
        lambda row: get_gapfill_likelihoods(row['gapfill'], error_rate_dicts, possible_alleles[row['probe_idx']]), axis=1
    )

    for probe_idx in possible_alleles.keys():
        likelihood_keys = possible_alleles[probe_idx]
        for key in likelihood_keys:
            unique_gapfill_list.loc[unique_gapfill_list['probe_idx'] == probe_idx,key] = unique_gapfill_list.loc[unique_gapfill_list['probe_idx'] == probe_idx, 'likelihood'].apply(lambda x: x.get(key, None))
        
    unique_gapfill_list = unique_gapfill_list.drop(columns=['likelihood'])

    unique_gapfill_list.to_csv('output/likelihood_tables_' + lib + '/' + BC + '_gapfill_likelihoods.csv', index=False)

819118 UMIs found
Collapsing UMIs across probes, 819118 UMIs remaining (100.00%)
Filtering probe reads based on read threshold (0) and min percent supporting (0.9), 792315 UMIs remaining (96.73%)
Filtering cells based on min counts (0) and genes (0) in WTA
Filtering probe reads based on cell barcodes in adata, 755337 UMIs remaining (92.21%)
